# Tutorial 03 — Bayesian inference & the workflow

`fairfluids.analysis.bayesian` runs NumPyro NUTS on the **same** symbolic models, with priors you
supply at fit time. Two layers:

* **building blocks** — `BayesianDataset`, `PriorSet`, and `ff.fit_group(..., backend="mcmc")`;
* **`BayesianWorkflow`** — a facade that orchestrates prior exploration → MCMC → diagnostics →
  model comparison → plots → write-back.

Requires the `[bayesian]` extra (JAX / NumPyro / ArviZ). MCMC here is kept small for speed.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import numpy as np
import pandas as pd
import sympy as sp

# Absolute data paths so the notebook runs from anywhere.
ROOT = "/home/sga/Code/FAIRFluids"
DENSITY = os.path.join(ROOT, "transition_water", "data", "Density")
VISCOSITY = os.path.join(ROOT, "transition_water", "data", "Viscosity")

import numpyro
numpyro.set_host_device_count(2)

from fairfluids import FAIRFluidsDocument
from fairfluids.analysis import models as fm
from fairfluids.analysis import fit as ff
from fairfluids.analysis import bayesian as bayes
from fairfluids.analysis.bayesian import (
    BayesianDataset, BayesianWorkflow, PriorSet, UniformPriorSpec,
)

doc = FAIRFluidsDocument.model_validate_json(
    open(os.path.join(DENSITY, "WaterGlycerol_egorov_density.json")).read()
)
ds = BayesianDataset.from_documents(
    {"Glycerol + Water": doc}, property="density", features=["temperature"],
    group_by=["source_doi", "mole_fraction_water"], min_points=4, log_observation=True,
)
# Keep just a few groups so the tutorial runs quickly.
ds.groups[:] = ds.groups[:3]
print("groups used:", len(ds))
print("registered Bayesian models:", bayes.list_models())

/home/sga/Code/FAIRFluids/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


groups used: 3
registered Bayesian models: ['arrhenius', 'density_exp_poly', 'density_exp_poly_degC_anchored', 'density_exp_poly_t0', 'density_exp_poly_t0_anchored', 'density_exp_poly_t0_mean_centered', 'extended_arrhenius', 'litovitz', 'litovitz_extended', 'vft']


## 1. Priors

There are **no hidden prior presets** — you state them explicitly with a `PriorSet`. Each parameter
gets a typed `PriorSpec` (Uniform / Normal / HalfNormal / LogNormal / TruncatedNormal); the
`sigma_scale` sets the HalfNormal scale of the observation-noise term.

In [2]:
priors = PriorSet(
    parameters={
        "A1": UniformPriorSpec(low=-5e-3, high=5e-3),
        "A2": UniformPriorSpec(low=-7e-6, high=1.45e-5),
    },
    sigma_scale=5e-4,
)
print(priors)

parameters={'A1': UniformPriorSpec(kind='uniform', low=-0.005, high=0.005), 'A2': UniformPriorSpec(kind='uniform', low=-7e-06, high=1.45e-05)} sigma_scale=0.0005 likelihood='normal' student_t_df=4.0


## 2. Frequentist vs Bayesian on one group — they agree

`ff.fit_group` drives either backend off the same model object. The NUTS posterior median lands on
the least-squares optimum (same maths, no drift).

In [3]:
rho_model = fm.get_model("density_exp_poly_t0_mean_centered")
group = max(ds.groups, key=lambda g: g.n_points)

# The builtin model calls its feature ``T``; the dataset column is ``temperature``.
fmap = {"T": "temperature"}
lsq = ff.fit_group(rho_model, group, feature_map=fmap)
mcmc = ff.fit_group(rho_model, group, backend="mcmc", feature_map=fmap, priors=priors,
                    num_warmup=400, num_samples=400, num_chains=1, seed=0)
post = mcmc.get_samples()
print("LSQ  A1=%.4e  A2=%.4e" % (lsq.values()["A1"], lsq.values()["A2"]))
print("NUTS A1=%.4e  A2=%.4e" % (float(np.median(post["A1"])), float(np.median(post["A2"]))))

LSQ  A1=3.6862e-04  A2=4.6291e-07
NUTS A1=3.6636e-04  A2=4.7213e-07


## 3. The workflow facade

`BayesianWorkflow.from_names` ties a dataset, a list of model names, and priors together. From there
the whole lifecycle is method calls.

In [4]:
wf = BayesianWorkflow.from_names(
    ds, ["density_exp_poly_t0_mean_centered"], priors=priors,
)

# Phase 1 — prior predictive sanity check (quantiles per model)
wf.explore_priors(n_samples=2000, seed=0)

,reference_group,model,n_samples,n_points,q01,q05,q50,q95,q99
0,https://doi.org/10.1016/j.tca.2013.07.012 | 0....,density_exp_poly_t0_mean_centered,2000,8,6.874979,6.984362,7.132612,7.265912,7.341468


In [5]:
# Phase 2 — fit every (model, group) pair
fit = wf.fit(num_warmup=400, num_samples=400, num_chains=2, seed=0)
print("fitted pairs:", len(fit.fits))

fitted pairs: 3


### Convergence diagnostics & posterior summary

In [6]:
wf.diagnostics().round(4).head()

,model,group_label,n_points,num_divergences,group_by_0,group_by_1,parameter,rhat,ess_bulk,ess_tail
0,density_exp_poly_t0_mean_centered,https://doi.org/10.1016/j.tca.2013.07.012 | 0....,8,0,https://doi.org/10.1016/j.tca.2013.07.012,0.0106,A1,1.0332,125.4556,127.5914
1,density_exp_poly_t0_mean_centered,https://doi.org/10.1016/j.tca.2013.07.012 | 0....,8,0,https://doi.org/10.1016/j.tca.2013.07.012,0.0106,A2,1.0325,126.2321,123.5629
2,density_exp_poly_t0_mean_centered,https://doi.org/10.1016/j.tca.2013.07.012 | 0....,8,0,https://doi.org/10.1016/j.tca.2013.07.012,0.0106,model_sigma,1.0284,161.9417,223.8561
3,density_exp_poly_t0_mean_centered,https://doi.org/10.1016/j.tca.2013.07.012 | 0....,8,0,https://doi.org/10.1016/j.tca.2013.07.012,0.0208,A1,1.0045,170.5165,139.5952
4,density_exp_poly_t0_mean_centered,https://doi.org/10.1016/j.tca.2013.07.012 | 0....,8,0,https://doi.org/10.1016/j.tca.2013.07.012,0.0208,A2,1.0049,170.3729,137.3509


In [7]:
wf.posterior_summary().round(5).head(10)

,model,group_id,group_label,parameter,n_points,median,mean,q05,q95,err_minus_90,err_plus_90,std,group_by_0,group_by_1,source_doi,mole_fraction_water
0,density_exp_poly_t0_mean_centered,"(https://doi.org/10.1016/j.tca.2013.07.012, 0....",https://doi.org/10.1016/j.tca.2013.07.012 | 0....,A1,8,0.00037,0.00036,0.00027,0.00044,0.00010,0.00007,0.00005,https://doi.org/10.1016/j.tca.2013.07.012,0.01055,https://doi.org/10.1016/j.tca.2013.07.012,0.01055
1,density_exp_poly_t0_mean_centered,"(https://doi.org/10.1016/j.tca.2013.07.012, 0....",https://doi.org/10.1016/j.tca.2013.07.012 | 0....,A2,8,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,https://doi.org/10.1016/j.tca.2013.07.012,0.01055,https://doi.org/10.1016/j.tca.2013.07.012,0.01055
2,density_exp_poly_t0_mean_centered,"(https://doi.org/10.1016/j.tca.2013.07.012, 0....",https://doi.org/10.1016/j.tca.2013.07.012 | 0....,model_sigma,8,0.00013,0.00014,0.00007,0.00024,0.00006,0.00012,0.00006,https://doi.org/10.1016/j.tca.2013.07.012,0.01055,https://doi.org/10.1016/j.tca.2013.07.012,0.01055
3,density_exp_poly_t0_mean_centered,"(https://doi.org/10.1016/j.tca.2013.07.012, 0....",https://doi.org/10.1016/j.tca.2013.07.012 | 0....,A1,8,0.00037,0.00037,0.00030,0.00045,0.00007,0.00008,0.00005,https://doi.org/10.1016/j.tca.2013.07.012,0.02083,https://doi.org/10.1016/j.tca.2013.07.012,0.02083
4,density_exp_poly_t0_mean_centered,"(https://doi.org/10.1016/j.tca.2013.07.012, 0....",https://doi.org/10.1016/j.tca.2013.07.012 | 0....,A2,8,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,https://doi.org/10.1016/j.tca.2013.07.012,0.02083,https://doi.org/10.1016/j.tca.2013.07.012,0.02083
5,density_exp_poly_t0_mean_centered,"(https://doi.org/10.1016/j.tca.2013.07.012, 0....",https://doi.org/10.1016/j.tca.2013.07.012 | 0....,model_sigma,8,0.00012,0.00013,0.00007,0.00025,0.00005,0.00013,0.00006,https://doi.org/10.1016/j.tca.2013.07.012,0.02083,https://doi.org/10.1016/j.tca.2013.07.012,0.02083
6,density_exp_poly_t0_mean_centered,"(https://doi.org/10.1016/j.tca.2013.07.012, 0....",https://doi.org/10.1016/j.tca.2013.07.012 | 0....,A1,8,0.00036,0.00036,0.00026,0.00043,0.00010,0.00007,0.00005,https://doi.org/10.1016/j.tca.2013.07.012,0.04219,https://doi.org/10.1016/j.tca.2013.07.012,0.04219
7,density_exp_poly_t0_mean_centered,"(https://doi.org/10.1016/j.tca.2013.07.012, 0....",https://doi.org/10.1016/j.tca.2013.07.012 | 0....,A2,8,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,https://doi.org/10.1016/j.tca.2013.07.012,0.04219,https://doi.org/10.1016/j.tca.2013.07.012,0.04219
8,density_exp_poly_t0_mean_centered,"(https://doi.org/10.1016/j.tca.2013.07.012, 0....",https://doi.org/10.1016/j.tca.2013.07.012 | 0....,model_sigma,8,0.00012,0.00013,0.00007,0.00024,0.00006,0.00012,0.00006,https://doi.org/10.1016/j.tca.2013.07.012,0.04219,https://doi.org/10.1016/j.tca.2013.07.012,0.04219


## 4. Compare competing models

Fit two density models and let ArviZ rank them by LOO (leave-one-out cross-validation). The
comparison returns three tidy tables: per-group best, per-group-per-model, and a global ranking.

In [8]:
wf2 = BayesianWorkflow.from_names(
    ds,
    ["density_exp_poly_t0_mean_centered", "density_exp_poly_t0"],
    priors={
        "density_exp_poly_t0_mean_centered": priors,
        "density_exp_poly_t0": PriorSet(
            parameters={
                "A1": UniformPriorSpec(low=-5e-3, high=5e-3),
                "A2": UniformPriorSpec(low=-7e-6, high=1.45e-5),
                "rho0": UniformPriorSpec(low=900.0, high=1400.0),
            },
            sigma_scale=5e-4,
        ),
    },
)
wf2.fit(num_warmup=400, num_samples=400, num_chains=2, seed=0)
cmp = wf2.compare()
cmp.global_ranking.round(3)

/home/sga/Code/FAIRFluids/.venv/lib/python3.13/site-packages/arviz_stats/loo/helper_loo.py:1146: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.66 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/home/sga/Code/FAIRFluids/.venv/lib/python3.13/site-packages/arviz_stats/loo/helper_loo.py:1146: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.66 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/home/sga/Code/FAIRFluid

,model,mean_elpd,sum_elpd,n_groups,n_warnings,weight
0,density_exp_poly_t0_mean_centered,58.391,175.174,3,3,1.0
1,density_exp_poly_t0,57.261,171.783,3,3,0.0


## 5. Plots

The workflow exposes plotting facades (matplotlib). A couple of common ones:

In [9]:
import matplotlib
matplotlib.use("Agg")  # headless rendering for notebook execution

fig1, ax1 = wf.plot_dataset_overview()
fig1.suptitle("Dataset overview")
fig1.tight_layout()

fig2, ax2 = wf.plot_posterior_predictive(
    "density_exp_poly_t0_mean_centered", group_id=group.group_id, feature="temperature",
)
fig2.tight_layout()
print("rendered overview + posterior-predictive figures")

rendered overview + posterior-predictive figures


## 6. Write the posterior back into a FAIRFluids document

`fit_to_fairfluids_document` (or `workflow.to_fairfluids_document`) records each group's posterior
summary back into a document so the fit is FAIR-archived alongside the data.

In [10]:
from fairfluids.analysis.bayesian import fit_to_fairfluids_document

enriched = fit_to_fairfluids_document(fit, doc, point_estimate="median")
print("write-back returned a", type(enriched).__name__)

write-back returned a FAIRFluidsDocument


## Recap

* Priors are **explicit** (`PriorSet` + typed `PriorSpec`s) — no hidden defaults.
* `ff.fit_group(..., backend="mcmc")` runs NUTS on the same model the least-squares path uses; the
  posterior median matches the LSQ optimum.
* `BayesianWorkflow` orchestrates the full lifecycle: `explore_priors` → `fit` → `diagnostics` /
  `posterior_summary` → `compare` → `plot_*` → `to_fairfluids_document`.
* Model comparison uses LOO and returns ready-to-read ranking tables.

**Next:** `04_fluid_selectors.ipynb` — slice multi-system documents cleanly.